In [1]:
import pandas as pd
import numpy as np
import random
import time

In [2]:
%matplotlib notebook

In [1]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from IPython.display import HTML
import os

class BatchProcessingMachine:
    def __init__(self, processing_time, batch_size):
        self.processing_time = processing_time
        self.batch_size = batch_size
        self.batch = []
        self.finish_time = 0
        self.busy = False
        self.event_log = []

    def process(self, current_time):
        if len(self.batch) == self.batch_size:
            self.busy = True
            self.finish_time = current_time + self.processing_time
            processed_product = self.batch[0]
            self.event_log.append((processed_product, current_time, self.finish_time, self.batch_size))
            self.batch = []
            return processed_product
        return None

    def add_to_batch(self, product, current_time):
        if not self.busy:
            if self.batch and self.batch[0] != product:
                # If a different product is added, clear the existing batch and reset (this should not happen by design)
                self.batch.clear()
            self.batch.append(product)
            if len(self.batch) == self.batch_size:
                return self.process(current_time)
        return None

    def update(self, current_time):
        if current_time >= self.finish_time and self.busy:
            self.busy = False
            return True
        return False

    def print_event_log(self):
        for event in self.event_log:
            print(f"Batch of Product {event[0]} processed from {event[1]} to {event[2]}")


class Machine:
    def __init__(self, processing_times, setup_times):
        self.processing_times = processing_times
        self.setup_times = setup_times
        self.current_product = None
        self.finish_time = 0
        self.busy = False
        self.event_log = []

    def process(self, product, current_time):
        setup_time = self.setup_times[self.current_product][product] if self.current_product is not None else 0
        self.current_product = product
        start_time = current_time + setup_time
        self.finish_time = start_time + self.processing_times[product]
        self.busy = True
        self.event_log.append((product, start_time, self.finish_time))
        return self.finish_time

    def update(self, current_time):
        if current_time >= self.finish_time and self.busy:
            self.busy = False
            return self.current_product
        return None

    def print_event_log(self):
        for event in self.event_log:
            print(f"Product {event[0]} processed from {event[1]} to {event[2]}")


class ProductionLine:
    def __init__(self, batch_machine_time, machine1_times, machine1_setups, machine2_times, machine2_setups, arrival_rate1, arrival_rate2):
        self.machine0 = BatchProcessingMachine(batch_machine_time, 4)
        self.machine1 = Machine(machine1_times, machine1_setups)
        self.machine2 = Machine(machine2_times, machine2_setups)
        self.arrival_rate1 = arrival_rate1
        self.arrival_rate2 = arrival_rate2
        self.next_arrival1 = 0
        self.next_arrival2 = 0
        self.queue = []
        self.finished_products = [0, 0]
        self.queue_log = []
        self.buffer = []
        self.buffer_log = []  # Log to track buffer status over time

    def add_item_to_queue0(self, product, current_time):
        self.queue0.append((product, current_time))
        self.log_queue0(current_time)

    def log_queue0(self, current_time):
        queue_size_0 = sum(1 for p, _ in self.queue0 if p == 0)
        queue_size_1 = sum(1 for p, _ in self.queue0 if p == 1)
        self.queue_log.append(((queue_size_0, queue_size_1), current_time))

    def add_item(self, product, current_time):
        self.queue.append((product, current_time))
        queue_size_0 = sum(1 for p, _ in self.queue if p == 0)
        queue_size_1 = sum(1 for p, _ in self.queue if p == 1)
        self.queue_log.append(((queue_size_0, queue_size_1), current_time))  # Log queue size per product and time        

    def run_step(self, current_time):
        if current_time >= self.next_arrival1:
            self.queue0.append((0, current_time))
            self.queue_log0.append((len(self.queue0), current_time))
            self.next_arrival1 += self.arrival_rate1

        if current_time >= self.next_arrival2:
            self.queue0.append((1, current_time))
            self.queue_log0.append((len(self.queue0), current_time))
            self.next_arrival2 += self.arrival_rate2

        if self.queue0 and not self.machine0.busy:
            product, _ = self.queue0.pop(0)
            processed_product = self.machine0.add_to_batch(product, current_time)
            if processed_product is not None and not self.machine1.busy:
                self.machine1.process(processed_product, current_time)

        if self.machine0.update(current_time):
            # Process next batch if machine0 just became free and queue is not empty
            while self.queue0 and not self.machine0.busy:
                product, _ = self.queue0.pop(0)
                processed_product = self.machine0.add_to_batch(product, current_time)
                if processed_product is not None:
                    if not self.machine1.busy:
                        self.machine1.process(processed_product, current_time)
                    break

        if self.queue and not self.machine1.busy:
            product, _ = self.queue.pop(0)
            self.machine1.process(product, current_time)

        if not self.machine2.busy and self.buffer:
            next_product = self.buffer.pop(0)
            self.machine2.process(next_product, current_time)
            self.log_buffer_status(current_time)  # Log buffer status after moving item to Machine 2

        if self.machine1.busy and current_time >= self.machine1.finish_time:
            finished_product = self.machine1.update(current_time)
            if finished_product is not None:
                if not self.machine2.busy:
                    self.machine2.process(finished_product, current_time)
                else:
                    self.buffer.append(finished_product)
                    self.log_buffer_status(current_time)  # Log buffer status after adding item

        finished_product = self.machine2.update(current_time)
        if finished_product is not None:
            self.finished_products[finished_product] += 1

        if current_time >= self.next_arrival1:
            self.add_item(0, current_time)
            self.next_arrival1 += self.arrival_rate1

        if current_time >= self.next_arrival2:
            self.add_item(1, current_time)
            self.next_arrival2 += self.arrival_rate2

    def log_buffer_status(self, current_time):
        # Count each product type in the buffer
        buffer_size_0 = sum(1 for p in self.buffer if p == 0)
        buffer_size_1 = sum(1 for p in self.buffer if p == 1)
        # Record current buffer state (counts of each product and current time)
        self.buffer_log.append(((buffer_size_0, buffer_size_1), current_time))


    def print_queue_status(self, current_time):
        queue_status = f"Queue at Time {current_time}: ["
        queue_status += ", ".join(f"Product {prod} (Arrived at {time})" for prod, time in self.queue)
        queue_status += "]"
        print(queue_status)

    def print_logs(self):
        print("Queue0 Event Log:")
        for log in self.queue_log:
            print(f"Time {log[1]}: Queue Sizes - Product 0: {log[0][0]}, Product 1: {log[0][1]}")

        print("\nMachine0 Batch Processing Log:")
        self.machine0.print_event_log()

        print("\nMachine1 Processing Log:")
        self.machine1.print_event_log()

        print("\nMachine2 Processing Log:")
        self.machine2.print_event_log()

        print("\nBuffer Status Log:")
        for log in self.buffer_log:
            print(f"Time {log[1]}: Buffer Sizes - Product 0: {log[0][0]}, Product 1: {log[0][1]}")

# Simulation setup
production_line = ProductionLine(10, {0: 3, 1: 4}, {None: {0: 5, 1: 3}, 0: {0: 0, 1: 10}, 1: {0: 5, 1: 0}},
                                 {0: 10, 1: 12}, {None: {0: 0, 1: 0}, 0: {0: 0, 1: 2}, 1: {0: 3, 1: 0}}, 20, 10)

# Run the production line for a specific period
for current_time in range(100):
    production_line.run_step(current_time)

# Print event logs of each machine and the buffer
production_line.print_logs()



AttributeError: 'ProductionLine' object has no attribute 'queue0'

In [ ]:
#Next steps
# 1) Introduce a demand forecast into the state space and reward it for meeting future excess demand
# 2) See if it can switch demand patterns quickly 
# 3) Q-learning implementaion to achive the same result
# 4) Write a LP program of this